In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Setup paths
base_dir = Path.cwd().parent
processed_data_dir = base_dir / 'data' / 'processed'
models_dir = base_dir / 'models' / 'text_encoder'
models_dir.mkdir(parents=True, exist_ok=True)

print("Loading Master Text Dataset...")
df = pd.read_csv(processed_data_dir / 'master_text_data.csv')

# Drop missing values to prevent neural network failure
df = df.dropna(subset=['text'])

# ENCODE LABELS (Fixing the Error using your Original Logic)
print("Encoding Labels...")
label_mapping = {'Safe': 0, 'Web_Attack': 1, 'LLM_Attack': 2}
df['target'] = df['label'].map(label_mapping)

X = df['text'].values
y = df['target'].values # Using the encoded numerical target

print(f"Dataset loaded successfully. Total samples: {len(df)}")

Loading Master Text Dataset...
Encoding Labels...
Dataset loaded successfully. Total samples: 81729


In [2]:
import time
import gc
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

print("--- 1. Text Vectorization & Splitting ---")
start_t = time.time()

# Memory Efficient TF-IDF Vectorization
print("Converting Text to Numerical Vectors (Float32)...")
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), dtype=np.float32, min_df=3)

# Convert all text to vectors (keeping it sparse temporarily to save RAM)
X_sparse = tfidf.fit_transform(X)

print("Preparing Neural Network Data (Train/Test Split)...")
# Split while still sparse
X_train_sparse, X_test_sparse, y_train, y_test = train_test_split(
    X_sparse, y, test_size=0.2, random_state=42, stratify=y
)

print("Converting to Dense Arrays for Deep Learning...")
# Safely convert to dense arrays now
X_train_vec = X_train_sparse.toarray()
X_test_vec = X_test_sparse.toarray()

# Clear memory
del X_sparse
gc.collect()

print(f"Done in {time.time() - start_t:.2f} seconds. Input Shape: {X_train_vec.shape}")

--- 1. Text Vectorization & Splitting ---
Converting Text to Numerical Vectors (Float32)...
Preparing Neural Network Data (Train/Test Split)...
Converting to Dense Arrays for Deep Learning...
Done in 10.95 seconds. Input Shape: (65383, 5000)


In [3]:
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.utils import to_categorical

print("--- 2. Label Encoding & Firewall Weights ---")

print("Calculating Class Weights (With Penalty Capping)...")
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)

# 💥 The Master Fix: Cap the LLM penalty to 15.0 to prevent False Positives!
weight_dict = {0: weights[0], 1: weights[1], 2: min(weights[2], 15.0)}
print(f"Applied Weights: Safe={weight_dict[0]:.2f}, Web={weight_dict[1]:.2f}, LLM={weight_dict[2]:.2f}")

print("Converting labels to categorical format... ")
y_train_cat = to_categorical(y_train, num_classes=3)
y_test_cat = to_categorical(y_test, num_classes=3)

print("Pipeline Ready for Neural Network Training!")

--- 2. Label Encoding & Firewall Weights ---
Calculating Class Weights (With Penalty Capping)...
Applied Weights: Safe=0.39, Web=2.32, LLM=15.00
Converting labels to categorical format... 
Pipeline Ready for Neural Network Training!


In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

print("Building Deep Neural Network Architecture...")

model = Sequential([
    Dense(256, activation='relu', input_shape=(X_train_vec.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(64, activation='relu'),
    Dense(3, activation='softmax') # 3 Output classes (Safe, Web, LLM)
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

Building Deep Neural Network Architecture...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │     1,280,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,323,139 (5.05 MB)

 Trainable params: 1,322,371 (5.04 MB)

 Non-trainable params: 768 (3.00 KB)

In [5]:
import numpy as np
import joblib
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report

print("Training the Deep NLP Engine with Capped Penalties...")
history = model.fit(
    X_train_vec, 
    y_train_cat, 
    epochs=10, 
    batch_size=128, 
    validation_split=0.2, 
    class_weight=weight_dict
)

print("\nEvaluating model on test data...")
y_pred_probs = model.predict(X_test_vec)

y_pred_custom = []
for probs in y_pred_probs:
    prob_safe, prob_web, prob_llm = probs[0], probs[1], probs[2]
    
    if prob_llm > 0.65:
        y_pred_custom.append(2)
    elif prob_web > 0.60:
        y_pred_custom.append(1)
    else:
        y_pred_custom.append(0)

y_pred_custom = np.array(y_pred_custom)
y_test_labels = np.argmax(y_test_cat, axis=1)

acc = accuracy_score(y_test_labels, y_pred_custom)

print(f"FINAL TEST ACCURACY: {acc*100:.4f}%")

print("Classification Report (With Firewall Thresholds):")
print(classification_report(y_test_labels, y_pred_custom, target_names=['Safe', 'Web_Attack', 'LLM_Attack']))

print("\nExporting Model and Vectorizer...")
models_dir = Path.cwd().parent / 'models' / 'text_encoder'
models_dir.mkdir(parents=True, exist_ok=True)

model.save(models_dir / 'dnn_text_model.h5')
joblib.dump(tfidf, models_dir / 'tfidf_vectorizer.pkl')

print("Model and vectorizer saved successfully.")

Training the Deep NLP Engine with Capped Penalties...
Epoch 1/10
409/409 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.9755 - loss: 0.1024 - val_accuracy: 0.9963 - val_loss: 0.0649
Epoch 2/10
409/409 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9936 - loss: 0.0406 - val_accuracy: 0.9945 - val_loss: 0.0257
Epoch 3/10
409/409 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9960 - loss: 0.0319 - val_accuracy: 0.9950 - val_loss: 0.0265
Epoch 4/10
409/409 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9967 - loss: 0.0292 - val_accuracy: 0.9953 - val_loss: 0.0399
Epoch 5/10
409/409 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9972 - loss: 0.0288 - val_accuracy: 0.9953 - val_loss: 0.0410
Epoch 6/10
409/409 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9966 - loss: 0.0307 - val_accuracy: 0.9954 - val_loss: 0.0290
Epoch 7/10
409/409 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9966 - loss: 0.0287 - val_accuracy: 0.9951 - val_loss: 0.0473
Epoch 8/10
409/409 ━━━━━━━━━━━━━━━━━━━━ 5s 

FINAL TEST ACCURACY: 99.5351%
Classification Report (With Firewall Thresholds):
              precision    recall  f1-score   support

        Safe       1.00      1.00      1.00     13941
  Web_Attack       1.00      0.99      0.99      2353
  LLM_Attack       0.52      0.83      0.64        52

    accuracy                           1.00     16346
   macro avg       0.84      0.94      0.88     16346
weighted avg       1.00      1.00      1.00     16346


Exporting Model and Vectorizer...
Model and vectorizer saved successfully.
